## 2. Training

After generating the training dataset *train_data.npz* we can move forward to train the model.

We will start with some utils needed to perform the training step. Then we will instatiate the model, specifying all the architectures of each module of the network and their respective hyperparameters.

As described in [Materials and Methods](https://www.nature.com/articles/s41377-023-01230-2#Sec9) the two generators $G_A$ and $G_B$ and the second stage deblurring network *DeblurNet*, share the same architecture ([Supplementary informations](https://www.nature.com/articles/s41377-023-01230-2#Sec34), *fig.S18*).

The two discriminators $D_A$ and $D_B$ differ from the previous ones, but are identical to each other.

### 2.1 Import packages

In [ ]:
import os
import sys
import torch

notebook_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(notebook_dir, '..'))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from model import Self_net_architecture
import torch.optim as optim             
from model import pytorch_dataset
from model import checkpoint
import itertools
from model.ssim import SSIM
from model.Charbonnier_loss import L1_Charbonnier_loss

### 2.2 Utils

#### 2.2.1 Learning rate adjustment

In [2]:
def adjust_lr(init_lr,optimizer,epoch,step,gamma):
    if (epoch+1)%step==0:
        times=(epoch+1)/step
        lr=init_lr*gamma**times
        for params in optimizer.param_groups:
            params['lr']=lr

#### 2.2.2 Backward step Discriminator

In [3]:
def backward_D_basic(lambda_gan,netD, criterionGAN,real, fake,type, device):
    """Calculate GAN loss for the discriminator
           Parameters:
               netD (network)      -- the discriminator D
               real (tensor array) -- real images
               fake (tensor array) -- images generated by a generator
           Return the discriminator loss.
           We also call loss_D.backward() to calculate the gradients.
           """
    # Real
    pred_real = netD(real)
    loss_D_real = criterionGAN(pred_real, True)

    # Fake
    pred_fake = netD(fake.detach())
    loss_D_fake =criterionGAN(pred_fake, False)

    # Combined loss and calculate gradients
    if type=='lsgan' or type=='vanilla':
        loss_D = lambda_gan*(loss_D_real + loss_D_fake) * 0.5
        loss_D.backward()
        return loss_D
    else:
        print('warning in calculating loss_D')

#### 2.2.3 Set requires grad for training

In [4]:
def set_requires_grad(net, requires_grad=False):
    for param in net.parameters():
        param.requires_grad = requires_grad

#### 2.2.4 Training definition

This is the main function to train the network. We define the training steps of all the newtorks, including the losses of that we want to minimize.

During the training, we will save model checkpoints and their respective output that will be useful in the next notebook. 

Specifically, the code (see **model/checkpoint.py**) retrieves the first slice of the XZ plane (**dataset/raw_data/xz**) and passes it through the model, saving the result in the **checkpoint/intermediate_results** folder. If you want to choose a different slice, please change the value of the `index` variable in `checkpoint.py`

So, if you want to check how the model training is going, you can refer to these outputs. You can open the reference slice and the outputs (both *.tiff* files) in Napari or ImageJ/Fiji.

For all the information about how the model is trained, what are the losses to minimize and which parameters they have, please refer to [Materials and Methods](https://www.nature.com/articles/s41377-023-01230-2#Sec9), and in particular to *Self-Net for isotropic restoration*, *Network architecture* and *Training and testing of Self-Net* sections

In [5]:
def train(train_set,netD_A,netD_B,netG_A,netG_B,deblur_net,hyperpars,device,criterionGAN,criterionL1,SSIM_loss,optimD,optimG,optim_deblur):
    lambda_cycle=1
    lambda_deblur=1
    lambda_feedback=0
    lambda_gan=0.1
    lambda_ssim=0.1

    netD_A.train()
    netD_B.train()

    netG_A.train()
    netG_B.train()
    deblur_net.train()

    fakeA_pool=pytorch_dataset.ImagePool(50)
    fakeB_pool=pytorch_dataset.ImagePool(50)

    Loss_D=0.0
    Loss_G_GAN=0.0
    Loss_G_cycle=0.0

    Loss_feedback=0.0
    Loss_deblur=0.0


    for epoch in range(hyperpars['epochs']):
        param1 = optimD.param_groups[0]
        param2=optimG.param_groups[0]
        adjust_lr(hyperpars['learning_rate_D'], optimD, epoch, 20, 0.5)    #adjust learning rate every 15 epochs, decay by a factor of 0.5. Change the parameters to adjust the learning rate schedule as needed.
        adjust_lr(hyperpars['learning_rate_G'], optimG, epoch, 20, 0.5)
        adjust_lr(hyperpars['learning_rate_G'], optim_deblur, epoch, 20, 0.5)

        print('epoch:{},learning_rate_D:{}'.format(epoch+1,param1['lr']))
        print('epoch:{},learning_rate_G:{}'.format(epoch + 1, param2['lr']))

        if epoch+1>=2:
            lambda_feedback=0.1

        for i, data in enumerate(train_set):

            #degradation modeling
            A=data['xy_lr']
            B=data['xz']
            C=data['xy']

            A=A.to(device)
            B=B.to(device)
            C=C.to(device)

            fake_B=netG_A(A)      #syn_xz

            fake_A=netG_B(B)

            set_requires_grad(deblur_net,False)
            fake_C1=deblur_net(fake_B)

            rec_A=netG_B(fake_B)

            rec_B=netG_A(fake_A)

            #optimize G
            set_requires_grad(netD_A, False)
            set_requires_grad(netD_B, False)

            optimG.zero_grad()

            # GAN loss D_A(G_A(A))
            loss_G_A = lambda_gan*criterionGAN(netD_A(fake_B), True)
            # GAN loss D_B(G_B(B))
            loss_G_B = lambda_gan*criterionGAN(netD_B(fake_A), True)


            # Forward cycle loss || G_B(G_A(A)) - A||
            loss_cycle_A = lambda_cycle*criterionL1(rec_A,A)+lambda_ssim*(1-SSIM_loss(rec_A,A))

            # Backward cycle loss || G_A(G_B(B)) - B||
            loss_cycle_B = lambda_cycle*criterionL1(rec_B,B)+lambda_ssim*(1-SSIM_loss(rec_B,B))

            loss_feedback = lambda_feedback*(criterionL1(fake_C1, C)+lambda_ssim*(1-SSIM_loss(fake_C1,C)))

            loss_G = loss_G_A + loss_G_B + loss_cycle_A + loss_cycle_B + loss_feedback

            loss_G.backward()
            optimG.step()

            #optimize D
            if (i+1)%2==0:
                set_requires_grad(netD_A, True)
                set_requires_grad(netD_B, True)

                optimD.zero_grad()

                fake_B = fakeB_pool.query(fake_B)

                loss_D_A =backward_D_basic(lambda_gan,netD_A, criterionGAN, B, fake_B, 'lsgan', device)

                fake_A = fakeA_pool.query(fake_A)
                loss_D_B = backward_D_basic(lambda_gan,netD_B,criterionGAN,A,fake_A,'lsgan',device)

                optimD.step()
            else:
                loss_D_A=torch.Tensor([0.0])
                loss_D_B=torch.Tensor([0.0])

            #Checkpoint
            Loss_D+=loss_D_A.item() + loss_D_B.item()
            Loss_G_GAN+=loss_G_A.item()+loss_G_B.item()

            Loss_feedback+=loss_feedback.item()

            Loss_G_cycle+=loss_cycle_A.item()+loss_cycle_B.item()

            if (i + 1) % hyperpars['log_interval'] == 0:
                print('epochs: {} {}/{}  degradation modeling stage: loss_D:{:4f} , '
                      'loss_G: {:4f} loss_G_cycle:{:4f} loss_feedback:{:4f}'.
                format(epoch+1,

                (i + 1) * hyperpars['batch_size'], len(train_set.dataset),
                Loss_D / (hyperpars['log_interval']*lambda_gan),Loss_G_GAN/(hyperpars['log_interval']*lambda_gan),
                Loss_G_cycle / (hyperpars['log_interval']*lambda_cycle),Loss_feedback/(hyperpars['log_interval'])))

                Loss_D = 0.0
                Loss_G_GAN = 0.0
                Loss_G_cycle = 0.0
                Loss_feedback=0.0

            #deblurring
            set_requires_grad(deblur_net, True)
            fake_B = netG_A(A)  # syn_lr_xy
            fake_C2 = deblur_net(fake_B.detach())

            optim_deblur.zero_grad()

            loss_deblur = lambda_deblur * criterionL1(fake_C2, C) + lambda_ssim * (1 - SSIM_loss(fake_C2, C))

            loss_deblur.backward()
            optim_deblur.step()

            # Checkpoint
            Loss_deblur += loss_deblur.item()

            if (i + 1) % hyperpars['log_interval'] == 0:
                print('epochs: {} {}/{}  deblurring stage :loss_deblur:{:4f}'.
                      format(epoch + 1,
                             (i + 1) * hyperpars['batch_size'], len(train_set.dataset),Loss_deblur /(hyperpars['log_interval']*lambda_deblur)))

                Loss_deblur = 0.0

            if (i + 1) % hyperpars['imshow_interval'] == 0:
                checkpoint.intermediate_output(deblur_net, device, hyperpars['path'], hyperpars['min_v'], hyperpars['max_v'], epoch, (i+1)*hyperpars['batch_size'])
                torch.save(deblur_net.state_dict(),hyperpars['path']+'/checkpoint/saved_models/' + 'deblur_net_{}_{}'.format(epoch + 1,(i+1)*hyperpars['batch_size']) + '.pkl')

### 2.3 Model train

In this section we will finally train the model. 

We will first define the parameters needed by the networks, we will instantiate those networks and then we will lunch the training.

Training time will depends on the number of epochs specified by the user. Please, to accelerate it use GPU.

**Note**: this part has been slightly modified since in their original code they were passing parameters through command line.

#### 2.3.1 Hyperparameters

In [ ]:
#Please change the hyperparameters and training settings as needed. The current settings are just examples and may not be optimal for your specific use case.
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

input_nc=1
output_nc=1

#Hyperparameters and training settings
hyperparameters = {
    'batch_size': 10,
    'epochs': 40,
    'path': parent_dir,        #path to the training data. Change this to the actual path of your training data.

    'min_v': 0, 
    'max_v': 255,

    'learning_rate_G': 1e-4,
    'learning_rate_D': 1e-4,
    'beta1': 0.7,               #beta1/beta2 for Adam optimizer. Change this to adjust the momentum term of the Adam optimizer for the generator and discriminator.
    'beta2': 0.99,

    'log_interval': 1,        #interval for logging training progress. Change this to adjust how often you want to see the training progress.
    'imshow_interval': 100,     #interval for saving intermediate outputs. Change this to adjust how often you want to save intermediate outputs.
}

checkpoint_path = os.path.join(hyperparameters['path'], 'checkpoint/')

if not os.path.exists(checkpoint_path):
    os.makedirs(checkpoint_path)

model_save_path = os.path.join(checkpoint_path, 'saved_models')
intermdiate_results_path = os.path.join(checkpoint_path, 'intermediate_results')

if not os.path.exists(model_save_path):
    os.makedirs(model_save_path)
if not os.path.exists(intermdiate_results_path):
    os.makedirs(intermdiate_results_path)

#### 2.3.2 Network definition and Training execution

In [7]:
netD_A = Self_net_architecture.define_D(input_nc=input_nc, ndf=64, netD='n_layers', n_layers_D=2, device=device, norm='instance')
netD_B = Self_net_architecture.define_D(input_nc=input_nc, ndf=64, netD='n_layers', n_layers_D=2, device=device, norm='instance')

netG_A = Self_net_architecture.define_G(input_nc=input_nc, output_nc=output_nc, ngf=64, netG='deblur_net', device=device,use_dropout=False,norm='instance')
netG_B = Self_net_architecture.define_G(input_nc=input_nc, output_nc=output_nc, ngf=64, netG='deblur_net', device=device,use_dropout=False,norm='instance')

deblur_net=Self_net_architecture.define_G(input_nc=input_nc, output_nc=output_nc, ngf=64, netG='deblur_net', device=device,use_dropout=False,norm='instance')

criterionGAN=Self_net_architecture.GANLoss(gan_mode='lsgan').to(device)

SSIM_loss = SSIM(data_range=255, size_average=True, win_size=11, win_sigma=1.5, channel=1)

criterionL1=L1_Charbonnier_loss()
optimG = optim.Adam(itertools.chain(netG_A.parameters(), netG_B.parameters()),lr=hyperparameters['learning_rate_G'], betas=(hyperparameters['beta1'], hyperparameters['beta2']))
optimD=optim.Adam(itertools.chain(netD_A.parameters(),netD_B.parameters()),lr=hyperparameters['learning_rate_D'], betas=(hyperparameters['beta1'], hyperparameters['beta2']))
optim_deblur=optim.Adam(deblur_net.parameters(),lr=hyperparameters['learning_rate_G'], betas=(hyperparameters['beta1'], hyperparameters['beta2']))

train_set=pytorch_dataset.create_train_data(hyperparameters['path']+'/dataset/train_data/',hyperparameters['batch_size'],hyperparameters['min_v'],hyperparameters['max_v'])
train(train_set,netD_A,netD_B,netG_A,netG_B,deblur_net,hyperparameters,device,criterionGAN,criterionL1,SSIM_loss,optimD,optimG,optim_deblur)
print('done training!')

initialize network with normal
initialize network with normal
initialize network with normal
initialize network with normal
initialize network with normal
done data preprocessing!
epoch:1,learning_rate_D:0.0001
epoch:1,learning_rate_G:0.0001
epochs: 1 10/22440  degradation modeling stage: loss_D:0.000000 , loss_G: 2.055937 loss_G_cycle:0.184528 loss_feedback:0.000000
epochs: 1 10/22440  deblurring stage :loss_deblur:0.091101
epochs: 1 20/22440  degradation modeling stage: loss_D:1.947600 , loss_G: 2.125158 loss_G_cycle:0.185689 loss_feedback:0.000000
epochs: 1 20/22440  deblurring stage :loss_deblur:0.084355
epochs: 1 30/22440  degradation modeling stage: loss_D:0.000000 , loss_G: 1.909494 loss_G_cycle:0.266513 loss_feedback:0.000000
epochs: 1 30/22440  deblurring stage :loss_deblur:0.122602
epochs: 1 40/22440  degradation modeling stage: loss_D:1.856874 , loss_G: 1.851517 loss_G_cycle:0.222728 loss_feedback:0.000000
epochs: 1 40/22440  deblurring stage :loss_deblur:0.112575
epochs: 1 